# Task 3 - SARIMA (Kaggle/Colab)
Run SARIMA for the three required areas and save predictions + metrics.

In [ ]:
# Environment-aware setup + low-RAM data loading
# This cell is designed to run in Colab/Kaggle without loading the full parquet into memory.
from pathlib import Path
import time
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pyarrow.dataset as ds

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
CITY_FILE = "city_traffic.parquet"

if IS_COLAB:
    # Mount Google Drive in Colab so files under MyDrive are visible.
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)


def resolve_data_path() -> Path:
    # Kaggle: find attached parquet under /kaggle/input.
    if IS_KAGGLE:
        matches = list(Path("/kaggle/input").rglob(CITY_FILE))
        if matches:
            return matches[0]

    # Colab: user stores parquet under MyDrive/Colab Notebooks.
    if IS_COLAB:
        colab_root = Path("/content/drive/MyDrive/Colab Notebooks")
        direct = colab_root / CITY_FILE
        if direct.exists():
            return direct
        matches = list(colab_root.rglob(CITY_FILE))
        if matches:
            return matches[0]

    raise FileNotFoundError(
        "city_traffic.parquet not found. Kaggle: attach dataset. "
        "Colab: upload under MyDrive/Colab Notebooks."
    )


DATA_PATH = resolve_data_path()
print(f"Using dataset: {DATA_PATH}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical column names expected by forecasting notebooks.
SQUARE_COL = "square_id"
TIME_COL = "time_interval"
TRAFFIC_COL = "internet_traffic"

# Training/test interval used by this task.
TEST_START = pd.Timestamp("2013-12-16 00:00:00", tz="UTC")
TEST_END = pd.Timestamp("2013-12-22 23:50:00", tz="UTC")
FAST_MODE = True
TRAIN_WINDOW_DAYS = 30
TRAIN_START = TEST_START - pd.Timedelta(days=TRAIN_WINDOW_DAYS)
TRAIN_END = TEST_START - pd.Timedelta(minutes=10)


def stream_top_square(parquet_path: Path, batch_size: int = 250_000) -> int:
    # Streaming aggregation avoids loading all rows into RAM.
    dataset = ds.dataset(str(parquet_path), format="parquet")
    scanner = dataset.scanner(columns=[SQUARE_COL, TRAFFIC_COL], batch_size=batch_size)

    totals = {}
    for batch in scanner.to_batches():
        pdf = batch.to_pandas()
        grouped = pdf.groupby(SQUARE_COL, sort=False)[TRAFFIC_COL].sum()
        for sid, val in grouped.items():
            sid_int = int(sid)
            totals[sid_int] = totals.get(sid_int, 0.0) + float(val)

    if not totals:
        raise ValueError("Unable to compute top square: no rows found in parquet")
    return max(totals, key=totals.get)


def build_compact_frame(parquet_path: Path, target_squares: list[int], batch_size: int = 200_000) -> pd.DataFrame:
    # Stream only the required squares and time window, then aggregate by 10-minute timestamp.
    dataset = ds.dataset(str(parquet_path), format="parquet")

    square_filter = ds.field(SQUARE_COL).isin(target_squares)
    if FAST_MODE:
        time_filter = (ds.field(TIME_COL) >= TRAIN_START.tz_localize(None)) & (ds.field(TIME_COL) <= TEST_END.tz_localize(None))
    else:
        time_filter = ds.field(TIME_COL) <= TEST_END.tz_localize(None)

    scanner = dataset.scanner(
        columns=[SQUARE_COL, TIME_COL, TRAFFIC_COL],
        filter=square_filter & time_filter,
        batch_size=batch_size,
    )

    agg = {}
    for batch in scanner.to_batches():
        pdf = batch.to_pandas()
        if pdf.empty:
            continue
        pdf[TIME_COL] = pd.to_datetime(pdf[TIME_COL], utc=True)
        grouped = pdf.groupby([SQUARE_COL, TIME_COL], sort=False)[TRAFFIC_COL].sum()
        for (sid, ts), val in grouped.items():
            key = (int(sid), pd.Timestamp(ts))
            agg[key] = agg.get(key, 0.0) + float(val)

    if not agg:
        raise ValueError("No rows found for selected squares/time range")

    rows = [
        {SQUARE_COL: sid, TIME_COL: ts, TRAFFIC_COL: val}
        for (sid, ts), val in agg.items()
    ]
    compact_df = pd.DataFrame(rows)
    compact_df = compact_df.sort_values([SQUARE_COL, TIME_COL]).reset_index(drop=True)
    return compact_df


TOP_SQUARE_ID = stream_top_square(DATA_PATH)
target_squares = [TOP_SQUARE_ID, 4159, 4556]
target_squares = list(dict.fromkeys([int(x) for x in target_squares]))
print(f"Target squares: {target_squares}")

df = build_compact_frame(DATA_PATH, target_squares)
print(f"Compact rows loaded for modeling: {len(df):,}")


def get_area_series(data: pd.DataFrame, square_id: int) -> pd.Series:
    s = data[data[SQUARE_COL] == square_id].sort_values(TIME_COL)
    series = s.set_index(TIME_COL)[TRAFFIC_COL].astype(float)

    # Enforce fixed 10-minute frequency for stable SARIMA date handling.
    series = series.resample("10min").mean()
    series = series.interpolate(limit_direction="both")
    return series


def split_train_test(series: pd.Series, fast_mode: bool = True):
    if fast_mode:
        train = series.loc[TRAIN_START:TRAIN_END]
    else:
        train = series.loc[:TRAIN_END]

    test = series.loc[TEST_START:TEST_END]
    return train.dropna(), test.dropna()

In [ ]:
# SARIMA training + evaluation
# This cell uses fast defaults to finish quickly under Colab RAM/time constraints.
def evaluate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {"MAE": mae, "MAPE": mape, "RMSE": rmse}

ORDER = (1, 1, 1)
SEASONAL_ORDER = (1, 0, 1, 144)
MAXITER = 35

sarima_results = []
for square_id in target_squares:
    series = get_area_series(df, int(square_id))
    train, test = split_train_test(series, fast_mode=FAST_MODE)

    if len(train) < 3 * 144 or len(test) == 0:
        print(f"Skipping square {square_id}: insufficient train/test points")
        continue

    start = time.perf_counter()
    model = SARIMAX(
        train,
        order=ORDER,
        seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fit = model.fit(disp=False, maxiter=MAXITER)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    forecast = fit.forecast(steps=len(test))
    infer_time = time.perf_counter() - start

    y_true = test.values
    y_pred = forecast.values
    metrics = evaluate_metrics(y_true, y_pred)
    metrics.update({"square_id": int(square_id), "train_seconds": train_time, "infer_seconds": infer_time})
    sarima_results.append(metrics)
    print(f"Square {square_id} done in {train_time:.1f}s (train), {infer_time:.2f}s (forecast)")

    out = pd.DataFrame({
        "time_interval": test.index,
        "y_true": y_true,
        "y_pred": y_pred,
        "model": "SARIMA",
        "square_id": int(square_id),
    })
    out.to_csv(OUTPUT_DIR / f"sarima_{int(square_id)}.csv", index=False)

metrics_df = pd.DataFrame(sarima_results)
metrics_df.to_csv(OUTPUT_DIR / "metrics_sarima.csv", index=False)
print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
metrics_df